# VLM Candidate Classifier

This notebook trains a lightweight classifier on top of a frozen vision-language model for fixed grasp candidate selection.

## Model choice
- Primary path: frozen `CLIP` or `SigLIP` encoder + small binary head.
- `SigLIP` is useful here if you want a stronger vision-language backbone with a similar frozen-feature workflow.
- `QLoRA` is **not** the right primary tool for this stage, because we are not fine-tuning a large generative VLM. It would add complexity without helping the current candidate classification setup.

## Input
- image
- instruction
- candidate text

## Output
- binary label: should this candidate be selected?

We also evaluate episode-level top-1 accuracy over the 6 candidates for each episode.

In [ ]:
# Optional install cell.
# Run this once if transformers is missing in your environment.

# import sys
# !{sys.executable} -m pip install transformers accelerate

In [2]:
import sys
!{sys.executable} -m pip install transformers accelerate


from pathlib import Path
import json
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import AutoModel, AutoProcessor

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 11.4 MB/s eta 0:00:001m11.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 785.1/785.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 11.5 MB/s eta 0:00:0031m11.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 11.4 MB/s eta 0:00:0031m11.8 MB/s eta 0:00:01


device(type='cuda')

In [3]:
PROJECT_ROOT = Path('/home/gyanig/catkin_ws/src/tabletop_workspace_opt')
DATA_ROOT = PROJECT_ROOT / 'data' / 'milk_candidate_cls'
SAMPLES_PATH = DATA_ROOT / 'candidate_samples.jsonl'
SPLIT_ROOT = DATA_ROOT / 'splits'
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'openai/clip-vit-base-patch32'  # Change to 'google/siglip-base-patch16-224' if desired.
BATCH_SIZE = 8
EPOCHS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
IMAGE_SIZE_NOTE = 'Use processor defaults for the selected VLM.'

assert SAMPLES_PATH.exists(), f'Missing samples file: {SAMPLES_PATH}'

In [4]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

rows = load_jsonl(SAMPLES_PATH)
df = pd.DataFrame(rows)
df.head(3)

,sample_id,episode_id,scene_id,view_id,image_path,instruction,correct_candidate_id,label,task_type,scene_notes,episode_notes,slot_assignment,candidate_id,object_short,object_name,grasp_type,grasp_description,task_suitability,candidate_text
0,ep_milk_scene_01_top_1__whole_top,ep_milk_scene_01_top_1,milk_scene_01_top,top,images/milk_scene_01_top.png,Pick up the whole milk.,whole_top,1,pickup,fixed layout v1,,"{'left': 'whole_milk', 'center': 'oat_milk', '...",whole_top,whole,whole_milk,top,top grasp,pickup,Object: whole_milk. Grasp: top grasp. Task sui...
1,ep_milk_scene_01_top_1__whole_side,ep_milk_scene_01_top_1,milk_scene_01_top,top,images/milk_scene_01_top.png,Pick up the whole milk.,whole_top,0,pickup,fixed layout v1,,"{'left': 'whole_milk', 'center': 'oat_milk', '...",whole_side,whole,whole_milk,side,side grasp,pour,Object: whole_milk. Grasp: side grasp. Task su...
2,ep_milk_scene_01_top_1__oat_top,ep_milk_scene_01_top_1,milk_scene_01_top,top,images/milk_scene_01_top.png,Pick up the whole milk.,whole_top,0,pickup,fixed layout v1,,"{'left': 'whole_milk', 'center': 'oat_milk', '...",oat_top,oat,oat_milk,top,top grasp,pickup,Object: oat_milk. Grasp: top grasp. Task suita...


In [5]:
def base_scene_id(scene_id: str) -> str:
    for suffix in ('_top', '_side', '_lean'):
        if scene_id.endswith(suffix):
            return scene_id[: -len(suffix)]
    return scene_id

df['base_scene_id'] = df['scene_id'].map(base_scene_id)
df['text_input'] = df.apply(
    lambda row: f"Instruction: {row['instruction']}\nCandidate: {row['candidate_text']}",
    axis=1,
)

print('Rows:', len(df))
print('Episodes:', df['episode_id'].nunique())
print('Base scenes:', sorted(df['base_scene_id'].unique()))
print('Labels:', Counter(df['label']))

Rows: 258
Episodes: 43
Base scenes: ['milk_scene_01', 'milk_scene_02']
Labels: Counter({0: 215, 1: 43})


In [6]:
# Scene-based split.
# With two rounds recorded, a simple first pass is:
# - train on all but the last base scene
# - validate on the last base scene

base_scenes = sorted(df['base_scene_id'].unique())
assert len(base_scenes) >= 2, 'Need at least two base scenes for a scene-level split.'

train_scenes = base_scenes[:-1]
val_scenes = base_scenes[-1:]

train_df = df[df['base_scene_id'].isin(train_scenes)].reset_index(drop=True)
val_df = df[df['base_scene_id'].isin(val_scenes)].reset_index(drop=True)

print('Train base scenes:', train_scenes)
print('Val base scenes  :', val_scenes)
print('Train rows       :', len(train_df))
print('Val rows         :', len(val_df))

Train base scenes: ['milk_scene_01']
Val base scenes  : ['milk_scene_02']
Train rows       : 126
Val rows         : 132


In [7]:
train_path = SPLIT_ROOT / 'train_candidate_samples.jsonl'
val_path = SPLIT_ROOT / 'val_candidate_samples.jsonl'

train_df.to_json(train_path, orient='records', lines=True)
val_df.to_json(val_path, orient='records', lines=True)

print('Wrote:', train_path)
print('Wrote:', val_path)

Wrote: /home/gyanig/catkin_ws/src/tabletop_workspace_opt/data/milk_candidate_cls/splits/train_candidate_samples.jsonl
Wrote: /home/gyanig/catkin_ws/src/tabletop_workspace_opt/data/milk_candidate_cls/splits/val_candidate_samples.jsonl


## Dataset and Model

We freeze the VLM backbone and train only a small binary head. The head sees:
- image embedding
- text embedding
- elementwise product of image/text embeddings

In [8]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME).to(device)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False

def get_image_features(model, pixel_values):
    if hasattr(model, 'get_image_features'):
        return model.get_image_features(pixel_values=pixel_values)
    vision_outputs = model.vision_model(pixel_values=pixel_values)
    return vision_outputs.pooler_output

def get_text_features(model, input_ids, attention_mask):
    if hasattr(model, 'get_text_features'):
        return model.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
    text_outputs = model.text_model(input_ids=input_ids, attention_mask=attention_mask)
    return text_outputs.pooler_output

def l2_normalize(x):
    return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-6)

with torch.no_grad():
    dummy_img = Image.open((DATA_ROOT / df.iloc[0]['image_path'])).convert('RGB')
    dummy_inputs = processor(images=dummy_img, text=df.iloc[0]['text_input'], return_tensors='pt', padding=True)
    dummy_img_feat = get_image_features(backbone, dummy_inputs['pixel_values'].to(device))
    dummy_txt_feat = get_text_features(
        backbone,
        dummy_inputs['input_ids'].to(device),
        dummy_inputs['attention_mask'].to(device),
    )
feature_dim = int(dummy_img_feat.shape[-1])
feature_dim

512

In [10]:
class CandidateDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, image_root: Path):
        self.frame = frame.reset_index(drop=True)
        self.image_root = image_root

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(self.image_root / row['image_path']).convert('RGB')
        return {
            'image': image,
            'text': row['text_input'],
            'label': float(row['label']),
            'episode_id': row['episode_id'],
            'candidate_id': row['candidate_id'],
            'correct_candidate_id': row['correct_candidate_id'],
        }

def collate_fn(batch):
    images = [item['image'] for item in batch]
    texts = [item['text'] for item in batch]
    proc = processor(images=images, text=texts, return_tensors='pt', padding=True, truncation=True)
    proc['labels'] = torch.tensor([item['label'] for item in batch], dtype=torch.float32)
    proc['episode_ids'] = [item['episode_id'] for item in batch]
    proc['candidate_ids'] = [item['candidate_id'] for item in batch]
    proc['correct_candidate_ids'] = [item['correct_candidate_id'] for item in batch]
    return proc

train_ds = CandidateDataset(train_df, DATA_ROOT)
val_ds = CandidateDataset(val_df, DATA_ROOT)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

In [11]:
class FrozenVLMClassifier(nn.Module):
    def __init__(self, vlm_backbone, embed_dim):
        super().__init__()
        self.vlm_backbone = vlm_backbone
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        with torch.no_grad():
            image_feat = get_image_features(self.vlm_backbone, pixel_values)
            text_feat = get_text_features(self.vlm_backbone, input_ids, attention_mask)
            image_feat = l2_normalize(image_feat)
            text_feat = l2_normalize(text_feat)

        fused = torch.cat([image_feat, text_feat, image_feat * text_feat], dim=-1)
        logits = self.classifier(fused).squeeze(-1)
        return logits

model = FrozenVLMClassifier(backbone, feature_dim).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [12]:
def run_epoch(loader, train=True):
    model.train(train)
    all_labels = []
    all_probs = []
    all_episode_ids = []
    all_candidate_ids = []
    all_correct_ids = []
    total_loss = 0.0

    for batch in loader:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model(pixel_values, input_ids, attention_mask)
        loss = criterion(logits, labels)

        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        total_loss += float(loss.item()) * len(labels)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_probs.extend(probs.tolist())
        all_episode_ids.extend(batch['episode_ids'])
        all_candidate_ids.extend(batch['candidate_ids'])
        all_correct_ids.extend(batch['correct_candidate_ids'])

    pred_labels = [1 if p >= 0.5 else 0 for p in all_probs]
    acc = accuracy_score(all_labels, pred_labels)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, pred_labels, average='binary', zero_division=0)

    by_episode = defaultdict(list)
    for ep, cand, prob, correct in zip(all_episode_ids, all_candidate_ids, all_probs, all_correct_ids):
        by_episode[ep].append((cand, prob, correct))

    top1_correct = 0
    for ep, vals in by_episode.items():
        best_cand = sorted(vals, key=lambda x: x[1], reverse=True)[0][0]
        correct_cand = vals[0][2]
        if best_cand == correct_cand:
            top1_correct += 1
    episode_top1 = top1_correct / max(len(by_episode), 1)

    return {
        'loss': total_loss / max(len(all_labels), 1),
        'acc': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'episode_top1': episode_top1,
    }

In [13]:
history = []
best_val_top1 = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)

    row = {'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}}
    history.append(row)
    print(row)

    if val_metrics['episode_top1'] > best_val_top1:
        best_val_top1 = val_metrics['episode_top1']
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

history_df = pd.DataFrame(history)
history_df

{'epoch': 1, 'train_loss': 0.571286850505405, 'train_acc': 0.8015873015873016, 'train_precision': 0.25, 'train_recall': 0.09523809523809523, 'train_f1': 0.13793103448275862, 'train_episode_top1': 0.23809523809523808, 'val_loss': 0.4750347968303796, 'val_acc': 0.8333333333333334, 'val_precision': 0.0, 'val_recall': 0.0, 'val_f1': 0.0, 'val_episode_top1': 0.8636363636363636}
{'epoch': 2, 'train_loss': 0.4515157817375092, 'train_acc': 0.8333333333333334, 'train_precision': 0.0, 'train_recall': 0.0, 'train_f1': 0.0, 'train_episode_top1': 0.19047619047619047, 'val_loss': 0.4452572194012729, 'val_acc': 0.8333333333333334, 'val_precision': 0.0, 'val_recall': 0.0, 'val_f1': 0.0, 'val_episode_top1': 1.0}
{'epoch': 3, 'train_loss': 0.451439689076136, 'train_acc': 0.8333333333333334, 'train_precision': 0.0, 'train_recall': 0.0, 'train_f1': 0.0, 'train_episode_top1': 0.3333333333333333, 'val_loss': 0.44243128642891394, 'val_acc': 0.8333333333333334, 'val_precision': 0.0, 'val_recall': 0.0, 'val_f1

,epoch,train_loss,train_acc,train_precision,train_recall,train_f1,train_episode_top1,val_loss,val_acc,val_precision,val_recall,val_f1,val_episode_top1
0,1,0.571287,0.801587,0.25,0.095238,0.137931,0.238095,0.475035,0.833333,0.0,0.0,0.0,0.863636
1,2,0.451516,0.833333,0.00,0.000000,0.000000,0.190476,0.445257,0.833333,0.0,0.0,0.0,1.000000
2,3,0.451440,0.833333,0.00,0.000000,0.000000,0.333333,0.442431,0.833333,0.0,0.0,0.0,1.000000
3,4,0.447985,0.833333,0.00,0.000000,0.000000,0.095238,0.447379,0.833333,0.0,0.0,0.0,0.863636
4,5,0.444138,0.833333,0.00,0.000000,0.000000,0.238095,0.439365,0.833333,0.0,0.0,0.0,0.727273
5,6,0.446147,0.833333,0.00,0.000000,0.000000,0.095238,0.439507,0.833333,0.0,0.0,0.0,0.590909
6,7,0.439587,0.833333,0.00,0.000000,0.000000,0.333333,0.437137,0.833333,0.0,0.0,0.0,0.590909
7,8,0.435007,0.833333,0.00,0.000000,0.000000,0.380952,0.436084,0.833333,0.0,0.0,0.0,0.636364
8,9,0.440132,0.833333,0.00,0.000000,0.000000,0.190476,0.435490,0.833333,0.0,0.0,0.0,0.590909
9,10,0.440579,0.833333,0.00,0.000000,0.000000,0.238095,0.433327,0.833333,0.0,0.0,0.0,0.454545


In [14]:
if best_state is not None:
    model.load_state_dict(best_state)

save_dir = PROJECT_ROOT / 'outputs' / 'vlm_candidate_classifier'
save_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        'model_name': MODEL_NAME,
        'classifier_state_dict': model.classifier.state_dict(),
        'feature_dim': feature_dim,
        'train_scenes': train_scenes,
        'val_scenes': val_scenes,
    },
    save_dir / 'classifier.pt'
)
history_df.to_csv(save_dir / 'history.csv', index=False)
history_df

,epoch,train_loss,train_acc,train_precision,train_recall,train_f1,train_episode_top1,val_loss,val_acc,val_precision,val_recall,val_f1,val_episode_top1
0,1,0.571287,0.801587,0.25,0.095238,0.137931,0.238095,0.475035,0.833333,0.0,0.0,0.0,0.863636
1,2,0.451516,0.833333,0.00,0.000000,0.000000,0.190476,0.445257,0.833333,0.0,0.0,0.0,1.000000
2,3,0.451440,0.833333,0.00,0.000000,0.000000,0.333333,0.442431,0.833333,0.0,0.0,0.0,1.000000
3,4,0.447985,0.833333,0.00,0.000000,0.000000,0.095238,0.447379,0.833333,0.0,0.0,0.0,0.863636
4,5,0.444138,0.833333,0.00,0.000000,0.000000,0.238095,0.439365,0.833333,0.0,0.0,0.0,0.727273
5,6,0.446147,0.833333,0.00,0.000000,0.000000,0.095238,0.439507,0.833333,0.0,0.0,0.0,0.590909
6,7,0.439587,0.833333,0.00,0.000000,0.000000,0.333333,0.437137,0.833333,0.0,0.0,0.0,0.590909
7,8,0.435007,0.833333,0.00,0.000000,0.000000,0.380952,0.436084,0.833333,0.0,0.0,0.0,0.636364
8,9,0.440132,0.833333,0.00,0.000000,0.000000,0.190476,0.435490,0.833333,0.0,0.0,0.0,0.590909
9,10,0.440579,0.833333,0.00,0.000000,0.000000,0.238095,0.433327,0.833333,0.0,0.0,0.0,0.454545


## Notes for next iteration

- If CLIP works but seems weak, try `google/siglip-base-patch16-224` by changing `MODEL_NAME`.
- Keep the encoder frozen first. Only unfreeze later if the dataset becomes substantially larger.
- Add a rule-based and text-only baseline so you can show whether image features provide additional value.
- If you record more scenes, keep splitting by `base_scene_id`, not by row.